In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold


def bemis_murcko_scaffold(smiles: str) -> str | None:
    """Return Bemis–Murcko scaffold as SMILES. None if parsing fails."""
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    if scaf is None:
        return None
    return Chem.MolToSmiles(scaf, isomericSmiles=False)


def scaffold_split(df: pd.DataFrame, smiles_col="smiles", test_size=0.2, seed=42):
    """
    Split by Bemis–Murcko scaffolds: all compounds sharing a scaffold go to the same split.
    Returns: df_train, df_test
    """
    df = df.copy()
    df["scaffold"] = df[smiles_col].apply(bemis_murcko_scaffold)

    # Drop rows where scaffold couldn't be computed (or handle separately)
    df = df.dropna(subset=["scaffold"]).reset_index(drop=True)

    # Group indices by scaffold
    scaffold_to_indices = df.groupby("scaffold").indices
    scaffolds = list(scaffold_to_indices.keys())

    rng = np.random.default_rng(seed)
    rng.shuffle(scaffolds)

    n_total = len(df)
    n_test_target = int(round(test_size * n_total))

    test_scaffolds = []
    test_count = 0

    for scaf in scaffolds:
        idxs = scaffold_to_indices[scaf]
        if test_count + len(idxs) <= n_test_target:
            test_scaffolds.append(scaf)
            test_count += len(idxs)

    is_test = df["scaffold"].isin(test_scaffolds)
    df_train = df.loc[~is_test].drop(columns=["scaffold"]).reset_index(drop=True)
    df_test  = df.loc[ is_test].drop(columns=["scaffold"]).reset_index(drop=True)
    return df_train, df_test


# --- Beispiel ---
# df = pd.DataFrame({"smiles": [...], "y": [...]})
# train_df, test_df = scaffold_split(df, smiles_col="smiles", test_size=0.2, seed=42)
